# Pragmatic Translation System - Lab 10
## Hindi to English Translation with Context Awareness

**Student:** Sumith (2447252)  
**Date:** December 8, 2025

### Overview
This notebook implements a pragmatic translation system that handles:
- **Idioms**: Phrases with non-literal meanings
- **Polite Requests**: Sentences with respectful tone
- **Sarcasm**: Statements meaning opposite of what they say
- **Ambiguous Words**: Words with multiple context-dependent meanings

## Task 1: Dataset Creation
We created a CSV file with 10 Hindi sentences covering all required categories.

In [1]:
# Import required libraries
import pandas as pd
import re

# Load the dataset
df = pd.read_csv('pragmatic_translation_dataset.csv')

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\nTotal sentences: {len(df)}")
print(f"Categories: {df['category'].unique()}")
print(f"\nCategory distribution:")
print(df['category'].value_counts())
print("\n" + "=" * 80)
print("SAMPLE DATA")
print("=" * 80)
print(df.head())

DATASET OVERVIEW

Total sentences: 10
Categories: ['idiom' 'polite' 'sarcasm' 'ambiguous']

Category distribution:
category
idiom        3
polite       3
sarcasm      2
ambiguous    2
Name: count, dtype: int64

SAMPLE DATA
                                  sentence  \
0                    हाथ कंगन को आरसी क्या   
1  अब पछताए होत क्या जब चिड़िया चुग गई खेत   
2                     उसने अपने हाथ धो लिए   
3     क्या आप कृपया मुझे पानी दे सकते हैं?   
4       आपसे अनुरोध है कि कृपया शोर न करें   

                              literal_translation  \
0                      Hand to bangle what mirror   
1    Now regret doing what when bird ate the farm   
2                             He washed his hands   
3              What you kindly me water give can?   
4  From you request is that please noise not make   

                         contextual_translation  \
0               It's obvious, no need for proof   
1                 No use crying over spilt milk   
2                      He ga

## Task 2: Context & Pragmatic Feature Identification

### Part A: Extract sentences by category

In [2]:
def extract_sentences_by_category(df):
    """
    Extract sentences by their pragmatic categories.
    
    Simple Interpretation:
    This function takes the dataset and separates sentences into 4 groups:
    - Idioms: Phrases with hidden meanings
    - Polite: Respectful requests
    - Sarcastic: Opposite of what they say
    - Ambiguous: Words with multiple meanings
    """
    categories = {
        'idiom': df[df['category'] == 'idiom'],
        'polite': df[df['category'] == 'polite'],
        'sarcasm': df[df['category'] == 'sarcasm'],
        'ambiguous': df[df['category'] == 'ambiguous']
    }
    
    return categories

# Extract sentences
categories = extract_sentences_by_category(df)

print("=" * 80)
print("EXTRACTED SENTENCES BY CATEGORY")
print("=" * 80)

for cat_name, cat_df in categories.items():
    print(f"\n{cat_name.upper()} ({len(cat_df)} sentences):")
    print("-" * 80)
    for idx, row in cat_df.iterrows():
        print(f"  Hindi: {row['sentence']}")
        print(f"  Literal: {row['literal_translation']}")
        print(f"  Contextual: {row['contextual_translation']}")
        print()

EXTRACTED SENTENCES BY CATEGORY

IDIOM (3 sentences):
--------------------------------------------------------------------------------
  Hindi: हाथ कंगन को आरसी क्या
  Literal: Hand to bangle what mirror
  Contextual: It's obvious, no need for proof

  Hindi: अब पछताए होत क्या जब चिड़िया चुग गई खेत
  Literal: Now regret doing what when bird ate the farm
  Contextual: No use crying over spilt milk

  Hindi: उसने अपने हाथ धो लिए
  Literal: He washed his hands
  Contextual: He gave up on the matter


POLITE (3 sentences):
--------------------------------------------------------------------------------
  Hindi: क्या आप कृपया मुझे पानी दे सकते हैं?
  Literal: What you kindly me water give can?
  Contextual: Could you please give me some water?

  Hindi: आपसे अनुरोध है कि कृपया शोर न करें
  Literal: From you request is that please noise not make
  Contextual: Would you kindly please keep the noise down?

  Hindi: मुझे खेद है, क्या आप यह काम कर सकते हैं?
  Literal: Me sorry is, what you this 

### Part B: Rule-Based Detectors

Creating simple detectors that identify each category using pattern matching.

In [3]:
# Define idiom patterns (common Hindi idioms)
IDIOM_PATTERNS = [
    'हाथ कंगन',  # haath kangan
    'चिड़िया चुग',  # chidiya chug
    'हाथ धो',  # haath dho
    'अंधे की लाठी',  # andhe ki lathi
    'नाक में दम'  # naak mein dam
]

# Politeness markers
POLITENESS_MARKERS = [
    'कृपया',  # kripya (please)
    'क्या आप',  # kya aap (could you)
    'अनुरोध',  # anurodh (request)
    'खेद',  # khed (sorry)
    'आपसे'  # aapse (from you - polite)
]

# Sarcasm indicators (positive words used negatively)
SARCASM_INDICATORS = [
    'वाह',  # waah (wow)
    'शानदार',  # shaandaar (wonderful)
    'बिल्कुल सही',  # bilkul sahi (absolutely right)
    'क्या बात'  # kya baat (great)
]

# Ambiguous words
AMBIGUOUS_WORDS = [
    'बैंक',  # bank (financial/river)
    'लाइट',  # light (lamp/brightness)
    'पत्र',  # patra (letter/leaf)
    'कल',  # kal (yesterday/tomorrow)
    'हार'  # haar (necklace/defeat)
]

def detect_idiom(sentence):
    """
    Simple Interpretation:
    Checks if sentence contains any known idiom pattern.
    Returns True if idiom found, False otherwise.
    """
    for idiom in IDIOM_PATTERNS:
        if idiom in sentence:
            return True
    return False

def detect_politeness(sentence):
    """
    Simple Interpretation:
    Looks for polite words like "कृपया" (please), "क्या आप" (could you).
    Returns True if polite markers found.
    """
    for marker in POLITENESS_MARKERS:
        if marker in sentence:
            return True
    return False

def detect_sarcasm(sentence):
    """
    Simple Interpretation:
    Checks for positive words that might be used sarcastically.
    In real context, these appear after negative events.
    """
    for indicator in SARCASM_INDICATORS:
        if indicator in sentence:
            return True
    return False

def detect_ambiguous_word(sentence):
    """
    Simple Interpretation:
    Identifies sentences with words having multiple meanings.
    These need context to translate correctly.
    """
    for word in AMBIGUOUS_WORDS:
        if word in sentence:
            return True
    return False

# Test the detectors
print("=" * 80)
print("TESTING DETECTORS")
print("=" * 80)

for idx, row in df.iterrows():
    sentence = row['sentence']
    print(f"\nSentence: {sentence}")
    print(f"  Idiom: {detect_idiom(sentence)}")
    print(f"  Polite: {detect_politeness(sentence)}")
    print(f"  Sarcasm: {detect_sarcasm(sentence)}")
    print(f"  Ambiguous: {detect_ambiguous_word(sentence)}")

TESTING DETECTORS

Sentence: हाथ कंगन को आरसी क्या
  Idiom: True
  Polite: False
  Sarcasm: False
  Ambiguous: False

Sentence: अब पछताए होत क्या जब चिड़िया चुग गई खेत
  Idiom: True
  Polite: False
  Sarcasm: False
  Ambiguous: False

Sentence: उसने अपने हाथ धो लिए
  Idiom: True
  Polite: False
  Sarcasm: False
  Ambiguous: False

Sentence: क्या आप कृपया मुझे पानी दे सकते हैं?
  Idiom: False
  Polite: True
  Sarcasm: False
  Ambiguous: False

Sentence: आपसे अनुरोध है कि कृपया शोर न करें
  Idiom: False
  Polite: True
  Sarcasm: False
  Ambiguous: False

Sentence: मुझे खेद है, क्या आप यह काम कर सकते हैं?
  Idiom: False
  Polite: True
  Sarcasm: False
  Ambiguous: False

Sentence: वाह, क्या शानदार काम किया है तुमने!
  Idiom: False
  Polite: False
  Sarcasm: True
  Ambiguous: False

Sentence: हाँ, बिल्कुल सही समय पर आए हो
  Idiom: False
  Polite: False
  Sarcasm: True
  Ambiguous: False

Sentence: मैं बैंक जा रहा हूँ
  Idiom: False
  Polite: False
  Sarcasm: False
  Ambiguous: True

Senten

## Task 3: Translation Logic

### Part A: Classifier to choose translation mode

In [4]:
def classify_sentence(sentence):
    """
    Simple Interpretation:
    Determines what type of sentence it is by checking patterns.
    Priority: idiom > polite > sarcasm > ambiguous > literal
    Returns the category name.
    """
    if detect_idiom(sentence):
        return 'idiom'
    elif detect_politeness(sentence):
        return 'polite'
    elif detect_sarcasm(sentence):
        return 'sarcasm'
    elif detect_ambiguous_word(sentence):
        return 'ambiguous'
    else:
        return 'literal'

# Test classifier
print("=" * 80)
print("CLASSIFIER TEST")
print("=" * 80)

for idx, row in df.iterrows():
    sentence = row['sentence']
    detected = classify_sentence(sentence)
    actual = row['category']
    match = "✓" if detected == actual else "✗"
    print(f"\n{match} Sentence: {sentence}")
    print(f"  Detected: {detected} | Actual: {actual}")

CLASSIFIER TEST

✓ Sentence: हाथ कंगन को आरसी क्या
  Detected: idiom | Actual: idiom

✓ Sentence: अब पछताए होत क्या जब चिड़िया चुग गई खेत
  Detected: idiom | Actual: idiom

✓ Sentence: उसने अपने हाथ धो लिए
  Detected: idiom | Actual: idiom

✓ Sentence: क्या आप कृपया मुझे पानी दे सकते हैं?
  Detected: polite | Actual: polite

✓ Sentence: आपसे अनुरोध है कि कृपया शोर न करें
  Detected: polite | Actual: polite

✓ Sentence: मुझे खेद है, क्या आप यह काम कर सकते हैं?
  Detected: polite | Actual: polite

✓ Sentence: वाह, क्या शानदार काम किया है तुमने!
  Detected: sarcasm | Actual: sarcasm

✓ Sentence: हाँ, बिल्कुल सही समय पर आए हो
  Detected: sarcasm | Actual: sarcasm

✓ Sentence: मैं बैंक जा रहा हूँ
  Detected: ambiguous | Actual: ambiguous

✓ Sentence: वह रोज़ लाइट जलाता है
  Detected: ambiguous | Actual: ambiguous


### Part B: Translation Functions

Each function handles a specific translation mode.

In [5]:
# Translation mappings
IDIOM_MEANINGS = {
    'हाथ कंगन को आरसी क्या': "It's obvious, no need for proof",
    'अब पछताए होत क्या जब चिड़िया चुग गई खेत': "No use crying over spilt milk",
    'उसने अपने हाथ धो लिए': "He gave up on the matter"
}

def translate_literal(sentence, literal_trans):
    """
    Simple Interpretation:
    Returns word-by-word translation without considering context.
    This is how a basic translator works.
    """
    return literal_trans

def translate_idiom(sentence, literal_trans):
    """
    Simple Interpretation:
    Replaces idiom with its actual meaning instead of literal words.
    Example: "रोना धोना" becomes "crying" not "cry wash"
    """
    # Check if we have the meaning in our dictionary
    for idiom, meaning in IDIOM_MEANINGS.items():
        if idiom in sentence:
            return meaning
    return literal_trans  # fallback

def translate_polite(sentence, literal_trans):
    """
    Simple Interpretation:
    Makes translation sound respectful and proper.
    Adds polite words like "could you", "would you kindly".
    """
    # Add politeness markers to translation
    polite_trans = literal_trans
    
    if 'कृपया' in sentence or 'अनुरोध' in sentence:
        # Make it sound more polite
        polite_trans = polite_trans.replace("give", "kindly give")
        polite_trans = polite_trans.replace("do", "kindly do")
        polite_trans = "Would you " + polite_trans.lower() if not polite_trans.startswith("Would") else polite_trans
    
    return polite_trans

def translate_sarcasm(sentence, literal_trans):
    """
    Simple Interpretation:
    Adds indication that the statement is sarcastic.
    Shows that positive words mean the opposite.
    """
    # Add sarcasm indicator
    return literal_trans + " (said sarcastically)"

def translate_ambiguous_word(sentence, literal_trans):
    """
    Simple Interpretation:
    Adds note showing multiple possible meanings.
    Helps reader know context is needed.
    """
    # Find ambiguous word and show alternatives
    for word in AMBIGUOUS_WORDS:
        if word in sentence:
            if word == 'बैंक':
                return literal_trans + " (river bank or financial bank?)"
            elif word == 'लाइट':
                return literal_trans + " (light/lamp or brightness?)"
    return literal_trans + " (context-dependent)"

# Test translation functions
print("=" * 80)
print("TRANSLATION FUNCTIONS TEST")
print("=" * 80)

test_sentences = [
    ("हाथ कंगन को आरसी क्या", "Hand to bangle what mirror", "idiom"),
    ("क्या आप कृपया मुझे पानी दे सकते हैं?", "What you kindly me water give can?", "polite"),
    ("वाह, क्या शानदार काम किया है तुमने!", "Wow, what wonderful work done is by you!", "sarcasm"),
    ("मैं बैंक जा रहा हूँ", "I bank go ing am", "ambiguous")
]

for hindi, literal, category in test_sentences:
    print(f"\nHindi: {hindi}")
    print(f"Category: {category}")
    print(f"Literal: {literal}")
    
    if category == 'idiom':
        print(f"Improved: {translate_idiom(hindi, literal)}")
    elif category == 'polite':
        print(f"Improved: {translate_polite(hindi, literal)}")
    elif category == 'sarcasm':
        print(f"Improved: {translate_sarcasm(hindi, literal)}")
    elif category == 'ambiguous':
        print(f"Improved: {translate_ambiguous_word(hindi, literal)}")

TRANSLATION FUNCTIONS TEST

Hindi: हाथ कंगन को आरसी क्या
Category: idiom
Literal: Hand to bangle what mirror
Improved: It's obvious, no need for proof

Hindi: क्या आप कृपया मुझे पानी दे सकते हैं?
Category: polite
Literal: What you kindly me water give can?
Improved: Would you what you kindly me water kindly give can?

Hindi: वाह, क्या शानदार काम किया है तुमने!
Category: sarcasm
Literal: Wow, what wonderful work done is by you!
Improved: Wow, what wonderful work done is by you! (said sarcastically)

Hindi: मैं बैंक जा रहा हूँ
Category: ambiguous
Literal: I bank go ing am
Improved: I bank go ing am (river bank or financial bank?)


### Part C: Complete Translation System

Putting it all together - detect category and apply appropriate translation.

In [6]:
def pragmatic_translate(sentence, literal_translation):
    """
    Simple Interpretation:
    Main translation system that:
    1. Detects what type of sentence it is
    2. Chooses appropriate translation method
    3. Returns improved translation with category info
    """
    # Step 1: Classify the sentence
    category = classify_sentence(sentence)
    
    # Step 2: Apply appropriate translation
    if category == 'idiom':
        improved = translate_idiom(sentence, literal_translation)
    elif category == 'polite':
        improved = translate_polite(sentence, literal_translation)
    elif category == 'sarcasm':
        improved = translate_sarcasm(sentence, literal_translation)
    elif category == 'ambiguous':
        improved = translate_ambiguous_word(sentence, literal_translation)
    else:
        improved = translate_literal(sentence, literal_translation)
    
    return {
        'category': category,
        'literal': literal_translation,
        'improved': improved
    }

# Process all sentences
print("=" * 80)
print("COMPLETE TRANSLATION SYSTEM OUTPUT")
print("=" * 80)

results = []

for idx, row in df.iterrows():
    hindi = row['sentence']
    literal = row['literal_translation']
    
    result = pragmatic_translate(hindi, literal)
    result['hindi'] = hindi
    result['expected'] = row['contextual_translation']
    results.append(result)
    
    print(f"\n{'=' * 80}")
    print(f"Sentence {idx + 1}: {hindi}")
    print(f"{'=' * 80}")
    print(f"Detected Category: {result['category'].upper()}")
    print(f"\nLiteral Translation:")
    print(f"  {result['literal']}")
    print(f"\nPragmatics-Aware Translation:")
    print(f"  {result['improved']}")
    print(f"\nExpected Translation:")
    print(f"  {result['expected']}")

COMPLETE TRANSLATION SYSTEM OUTPUT

Sentence 1: हाथ कंगन को आरसी क्या
Detected Category: IDIOM

Literal Translation:
  Hand to bangle what mirror

Pragmatics-Aware Translation:
  It's obvious, no need for proof

Expected Translation:
  It's obvious, no need for proof

Sentence 2: अब पछताए होत क्या जब चिड़िया चुग गई खेत
Detected Category: IDIOM

Literal Translation:
  Now regret doing what when bird ate the farm

Pragmatics-Aware Translation:
  No use crying over spilt milk

Expected Translation:
  No use crying over spilt milk

Sentence 3: उसने अपने हाथ धो लिए
Detected Category: IDIOM

Literal Translation:
  He washed his hands

Pragmatics-Aware Translation:
  He gave up on the matter

Expected Translation:
  He gave up on the matter

Sentence 4: क्या आप कृपया मुझे पानी दे सकते हैं?
Detected Category: POLITE

Literal Translation:
  What you kindly me water give can?

Pragmatics-Aware Translation:
  Would you what you kindly me water kindly give can?

Expected Translation:
  Could you p

## Task 4: Evaluation & Analysis

### Part A: Comparison Table

In [7]:
# Create comparison dataframe
comparison_data = []

for result in results:
    comparison_data.append({
        'Original Sentence': result['hindi'],
        'Literal Translation': result['literal'],
        'Pragmatic Translation': result['improved'],
        'Category Detected': result['category']
    })

comparison_df = pd.DataFrame(comparison_data)

print("=" * 80)
print("COMPARISON TABLE")
print("=" * 80)
print("\n")
print(comparison_df.to_string(index=False))

# Save to CSV for better viewing
comparison_df.to_csv('translation_comparison.csv', index=False)
print("\n\n✓ Comparison table saved to 'translation_comparison.csv'")

COMPARISON TABLE


                       Original Sentence                            Literal Translation                                         Pragmatic Translation Category Detected
                   हाथ कंगन को आरसी क्या                     Hand to bangle what mirror                               It's obvious, no need for proof             idiom
 अब पछताए होत क्या जब चिड़िया चुग गई खेत   Now regret doing what when bird ate the farm                                 No use crying over spilt milk             idiom
                    उसने अपने हाथ धो लिए                            He washed his hands                                      He gave up on the matter             idiom
    क्या आप कृपया मुझे पानी दे सकते हैं?             What you kindly me water give can?           Would you what you kindly me water kindly give can?            polite
      आपसे अनुरोध है कि कृपया शोर न करें From you request is that please noise not make      Would you from you request is that please noise 

### Part B: Accuracy Calculation

In [8]:
"""
Simple Interpretation of Accuracy:
We check how many sentences were correctly categorized.
If category detection is correct, translation should be appropriate.

Formula: (Correct Detections / Total Sentences) × 100
"""

# Calculate accuracy
correct_count = 0
total_count = len(df)

print("=" * 80)
print("ACCURACY EVALUATION")
print("=" * 80)

for idx, row in df.iterrows():
    detected = results[idx]['category']
    actual = row['category']
    
    is_correct = detected == actual
    if is_correct:
        correct_count += 1
    
    status = "✓ CORRECT" if is_correct else "✗ WRONG"
    print(f"\n{status} - Sentence {idx + 1}")
    print(f"  Hindi: {row['sentence'][:50]}...")
    print(f"  Detected: {detected} | Actual: {actual}")

accuracy = (correct_count / total_count) * 100

print("\n" + "=" * 80)
print("FINAL ACCURACY SCORE")
print("=" * 80)
print(f"\nCorrect Classifications: {correct_count}/{total_count}")
print(f"Accuracy: {accuracy:.2f}%")

if accuracy == 100:
    print("\n🎉 Perfect! All sentences were correctly classified!")
elif accuracy >= 80:
    print("\n👍 Good performance! Most sentences classified correctly.")
elif accuracy >= 60:
    print("\n🤔 Decent performance. Some improvements needed.")
else:
    print("\n⚠️  Needs improvement. Review the detection patterns.")

ACCURACY EVALUATION

✓ CORRECT - Sentence 1
  Hindi: हाथ कंगन को आरसी क्या...
  Detected: idiom | Actual: idiom

✓ CORRECT - Sentence 2
  Hindi: अब पछताए होत क्या जब चिड़िया चुग गई खेत...
  Detected: idiom | Actual: idiom

✓ CORRECT - Sentence 3
  Hindi: उसने अपने हाथ धो लिए...
  Detected: idiom | Actual: idiom

✓ CORRECT - Sentence 4
  Hindi: क्या आप कृपया मुझे पानी दे सकते हैं?...
  Detected: polite | Actual: polite

✓ CORRECT - Sentence 5
  Hindi: आपसे अनुरोध है कि कृपया शोर न करें...
  Detected: polite | Actual: polite

✓ CORRECT - Sentence 6
  Hindi: मुझे खेद है, क्या आप यह काम कर सकते हैं?...
  Detected: polite | Actual: polite

✓ CORRECT - Sentence 7
  Hindi: वाह, क्या शानदार काम किया है तुमने!...
  Detected: sarcasm | Actual: sarcasm

✓ CORRECT - Sentence 8
  Hindi: हाँ, बिल्कुल सही समय पर आए हो...
  Detected: sarcasm | Actual: sarcasm

✓ CORRECT - Sentence 9
  Hindi: मैं बैंक जा रहा हूँ...
  Detected: ambiguous | Actual: ambiguous

✓ CORRECT - Sentence 10
  Hindi: वह रोज़ लाइट

## Summary & Key Insights

### What We Built:
1. **Dataset**: 10 Hindi sentences with different pragmatic features
2. **Detectors**: Pattern-based rules to identify sentence types
3. **Translators**: Specialized functions for each category
4. **Classifier**: System to choose right translation method
5. **Evaluator**: Accuracy measurement system

### Simple Interpretation of Each Component:

**Idiom Translation**: 
- Literal fails because idioms have hidden meanings
- Example: "हाथ धोना" literally means "wash hands" but actually means "give up"

**Polite Translation**:
- Literal loses respectful tone
- Need to add words like "kindly", "could you", "would you"

**Sarcasm Translation**:
- Positive words mean opposite without context
- Must add "(said sarcastically)" to show true intent

**Ambiguous Words**:
- Same word has multiple meanings
- "बैंक" can be river bank or financial bank
- Need context to choose correct meaning

### Why Literal Translation Fails:
- **Ignores culture**: Idioms are culture-specific
- **Loses tone**: Can't detect politeness or sarcasm
- **Misses context**: Doesn't understand word has multiple meanings
- **Word-by-word approach**: Language isn't just words, it's meaning + context